# 09 Train WLASL2000 Light V3 — Two-Stage Fine-Tuning

## What this notebook does

This notebook improves the current WLASL2000 model.

Current best result:

```text
WLASL2000 Light V2 Fine-tuned From WLASL1000
Top-1: 15.87%
Top-3: 35.08%
Top-5: 42.87%
Macro F1: 11.69%
```

Light V3 adds:

```text
1. Two-stage fine-tuning
2. Class-balanced focal loss
3. Warmup + cosine learning rate schedule
4. Slightly stronger safe augmentation
5. Temperature scaling for confidence calibration
6. Light V2 vs Light V3 comparison
```

# 1. Import libraries

In [ ]:
from pathlib import Path
import json, random, time, math, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

# 2. Set paths and configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL2000"
PREFIX = "wlasl2000"

BASE_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
CLEAN_INDEX_FILE = BASE_DIR / f"{PREFIX}_clean_keypoint_index.csv"
LABEL_MAP_FILE = PROJECT_ROOT / "data" / "label_maps" / DATASET_NAME / f"asl_{PREFIX}_labels.json"

MODEL_DIR = PROJECT_ROOT / "models" / "ASL" / DATASET_NAME
MODEL_DIR.mkdir(parents=True, exist_ok=True)

REPORT_DIR = PROJECT_ROOT / "reports" / f"phase1_{PREFIX}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

WLASL1000_MODEL_PATH = PROJECT_ROOT / "models" / "ASL" / "WLASL1000" / "bigru_attention_light_v2_wlasl1000.pt"

MODEL_NAME = "light_v3_two_stage_finetuned_from_wlasl1000"
MODEL_DISPLAY_NAME = "WLASL2000 Light V3 Two-Stage Fine-tuned From WLASL1000"
TRAINING_TITLE = "Be My Ear - WLASL2000 Light V3 Two-Stage Fine-tuning"

MODEL_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}.pt"
HISTORY_PATH = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_history.csv"
NORM_STATS_PATH = MODEL_DIR / f"{PREFIX}_{MODEL_NAME}_train_norm_stats.npz"
RESULT_FILE = MODEL_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"

INPUT_SIZE = 516
BASE_FEATURE_SIZE = 258
SEQUENCE_LENGTH = 60
USE_VELOCITY = True

HIDDEN_SIZE = 320
NUM_LAYERS = 2
DROPOUT = 0.35
BATCH_SIZE = 16

STAGE_1_EPOCHS = 10
STAGE_1_LR = 3e-4

STAGE_2_EPOCHS = 70
STAGE_2_LR = 5e-5

EARLY_STOPPING_PATIENCE = 16
WEIGHT_DECAY = 1e-4
GRADIENT_CLIP_NORM = 1.0

FOCAL_GAMMA = 1.5
LABEL_SMOOTHING = 0.03

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dataset:", DATASET_NAME)
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Clean index exists:", CLEAN_INDEX_FILE.exists(), CLEAN_INDEX_FILE)
print("Label map exists:", LABEL_MAP_FILE.exists(), LABEL_MAP_FILE)
print("WLASL1000 checkpoint exists:", WLASL1000_MODEL_PATH.exists(), WLASL1000_MODEL_PATH)
print("Model output:", MODEL_PATH)

# 3. Load clean WLASL2000 data

In [ ]:
df = pd.read_csv(CLEAN_INDEX_FILE)
NUM_CLASSES = df["label_id"].nunique()

print("Clean samples:", len(df))
print("Clean classes:", NUM_CLASSES)
print("Average samples per class:", round(len(df) / NUM_CLASSES, 2))

df.head()

# 4. Create safe train / validation / test split

In [ ]:
train_records, val_records, test_records, split_notes = [], [], [], []

for label_id, group in df.groupby("label_id"):
    group = group.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(group)

    if n == 1:
        train_records.append(group)
        rule = "train_only"
    elif n == 2:
        train_records.append(group.iloc[:1])
        test_records.append(group.iloc[1:])
        rule = "1_train_1_test"
    elif n == 3:
        train_records.append(group.iloc[:1])
        val_records.append(group.iloc[1:2])
        test_records.append(group.iloc[2:])
        rule = "1_train_1_val_1_test"
    else:
        n_test = max(1, int(round(n * 0.15)))
        n_val = max(1, int(round(n * 0.15)))
        test_records.append(group.iloc[:n_test])
        val_records.append(group.iloc[n_test:n_test + n_val])
        train_records.append(group.iloc[n_test + n_val:])
        rule = "standard_70_15_15"

    split_notes.append({"label_id": int(label_id), "samples": int(n), "split_rule": rule})

train_df = pd.concat(train_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat(val_records).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_records).sample(frac=1, random_state=SEED).reset_index(drop=True)

split_notes_df = pd.DataFrame(split_notes)
split_notes_file = BASE_DIR / f"{PREFIX}_light_v3_safe_split_notes.csv"
split_notes_df.to_csv(split_notes_file, index=False)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))
print("Train classes:", train_df["label_id"].nunique())
print("Validation classes:", val_df["label_id"].nunique())
print("Test classes:", test_df["label_id"].nunique())
print("Saved split notes:", split_notes_file)

split_notes_df["split_rule"].value_counts()

# 5. Compute train-only normalisation

In [ ]:
def compute_train_normalisation_stats(train_dataframe):
    total_sum = None
    total_sq_sum = None
    total_count = 0

    for path in tqdm(train_dataframe["keypoint_path"], desc="Computing train mean/std"):
        arr = np.load(path).astype(np.float32)

        if total_sum is None:
            total_sum = arr.sum(axis=0)
            total_sq_sum = (arr ** 2).sum(axis=0)
        else:
            total_sum += arr.sum(axis=0)
            total_sq_sum += (arr ** 2).sum(axis=0)

        total_count += arr.shape[0]

    mean = total_sum / total_count
    variance = (total_sq_sum / total_count) - (mean ** 2)
    variance = np.maximum(variance, 1e-6)
    std = np.sqrt(variance)

    return mean.astype(np.float32), std.astype(np.float32)

train_mean, train_std = compute_train_normalisation_stats(train_df)
np.savez(NORM_STATS_PATH, mean=train_mean, std=train_std)

print("Saved normalisation stats:", NORM_STATS_PATH)
print("Mean shape:", train_mean.shape)
print("Std shape:", train_std.shape)

# 6. Build dataset with Light V3 augmentation

In [ ]:
class WLASLKeypointDatasetLightV3(Dataset):
    def __init__(
        self,
        dataframe,
        mean,
        std,
        augment=False,
        noise_std=0.007,
        frame_mask_prob=0.04,
        temporal_shift_max=3,
        feature_dropout_prob=0.02,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.mean = mean.reshape(1, -1).astype(np.float32)
        self.std = std.reshape(1, -1).astype(np.float32)
        self.augment = augment
        self.noise_std = noise_std
        self.frame_mask_prob = frame_mask_prob
        self.temporal_shift_max = temporal_shift_max
        self.feature_dropout_prob = feature_dropout_prob

    def __len__(self):
        return len(self.dataframe)

    def _temporal_shift(self, keypoints):
        shift = np.random.randint(-self.temporal_shift_max, self.temporal_shift_max + 1)
        if shift == 0:
            return keypoints

        shifted = np.zeros_like(keypoints)
        if shift > 0:
            shifted[shift:] = keypoints[:-shift]
            shifted[:shift] = keypoints[0]
        else:
            shifted[:shift] = keypoints[-shift:]
            shifted[shift:] = keypoints[-1]
        return shifted

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        keypoints = np.load(row["keypoint_path"]).astype(np.float32)
        keypoints = (keypoints - self.mean) / (self.std + 1e-6)

        if self.augment:
            keypoints = self._temporal_shift(keypoints)

        velocity = np.zeros_like(keypoints, dtype=np.float32)
        velocity[1:] = keypoints[1:] - keypoints[:-1]

        features = np.concatenate([keypoints, velocity], axis=1).astype(np.float32)

        if self.augment:
            features += np.random.normal(0, self.noise_std, features.shape).astype(np.float32)

            frame_mask = np.random.rand(features.shape[0]) < self.frame_mask_prob
            features[frame_mask] = 0

            feature_mask = np.random.rand(features.shape[1]) < self.feature_dropout_prob
            features[:, feature_mask] = 0

        label = int(row["label_id"])
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

# 7. Create balanced data loaders

In [ ]:
train_dataset = WLASLKeypointDatasetLightV3(train_df, train_mean, train_std, augment=True)
val_dataset = WLASLKeypointDatasetLightV3(val_df, train_mean, train_std, augment=False)
test_dataset = WLASLKeypointDatasetLightV3(test_df, train_mean, train_std, augment=False)

class_counts = train_df["label_id"].value_counts().to_dict()
sample_weights = train_df["label_id"].map(lambda label: 1.0 / class_counts[label]).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

x_batch, y_batch = next(iter(train_loader))
print("Input batch shape:", x_batch.shape)
print("Label batch shape:", y_batch.shape)

# 8. Define Light V3 model

In [ ]:
class BiGRUAttentionLightV3(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2, dropout=0.35):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.gru = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
        )

        bi_hidden = hidden_size * 2

        self.attention = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

        self.classifier = nn.Sequential(
            nn.Linear(bi_hidden, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        x = self.input_projection(x)
        gru_out, _ = self.gru(x)

        attention_scores = self.attention(gru_out).squeeze(-1)
        attention_weights = torch.softmax(attention_scores, dim=1).unsqueeze(-1)
        context = torch.sum(gru_out * attention_weights, dim=1)

        return self.classifier(context)

def build_light_v3_model(num_classes):
    return BiGRUAttentionLightV3(
        input_size=INPUT_SIZE,
        hidden_size=HIDDEN_SIZE,
        num_classes=num_classes,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    )

# 9. Define class-balanced focal loss

In [ ]:
class ClassBalancedFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.999, gamma=1.5, label_smoothing=0.03):
        super().__init__()

        counts = torch.tensor(class_counts, dtype=torch.float32)
        effective_num = 1.0 - torch.pow(torch.tensor(beta), counts)
        weights = (1.0 - beta) / effective_num
        weights = weights / weights.mean()

        self.register_buffer("weights", weights)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        probs = torch.exp(log_probs)

        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.label_smoothing / (num_classes - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        focal_factor = torch.pow(1.0 - probs, self.gamma)
        class_weights = self.weights.to(logits.device).unsqueeze(0)

        loss = -true_dist * focal_factor * log_probs * class_weights
        loss = loss.sum(dim=1)

        return loss.mean()

train_class_counts = train_df.groupby("label_id").size().reindex(range(NUM_CLASSES), fill_value=1).values

criterion = ClassBalancedFocalLoss(
    class_counts=train_class_counts,
    beta=0.999,
    gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING,
).to(device)

print("Class-balanced focal loss ready.")

# 10. Load WLASL1000 Light V2 weights

In [ ]:
model = build_light_v3_model(NUM_CLASSES).to(device)

if not WLASL1000_MODEL_PATH.exists():
    raise FileNotFoundError(f"Cannot find WLASL1000 Light V2 checkpoint: {WLASL1000_MODEL_PATH}")

source_checkpoint = torch.load(WLASL1000_MODEL_PATH, map_location=device)
source_state = source_checkpoint["model_state_dict"]
target_state = model.state_dict()

loaded_keys = []
skipped_keys = []

for key, value in source_state.items():
    if key in target_state and target_state[key].shape == value.shape:
        target_state[key] = value
        loaded_keys.append(key)
    else:
        skipped_keys.append(key)

model.load_state_dict(target_state)

print("Loaded compatible WLASL1000 keys:", len(loaded_keys))
print("Skipped keys:", skipped_keys)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

# 11. Freeze / unfreeze helpers

In [ ]:
def freeze_feature_extractor(model):
    for name, param in model.named_parameters():
        param.requires_grad = False
        if name.startswith("classifier"):
            param.requires_grad = True

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

def count_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

freeze_feature_extractor(model)
print("Trainable parameters after freezing:", count_trainable_parameters(model))

# 12. Warmup + cosine learning rate helpers

In [ ]:
def set_optimizer_lr(optimizer, lr):
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

def cosine_lr(base_lr, current_step, total_steps, warmup_steps=0, min_lr_ratio=0.1):
    if current_step < warmup_steps:
        return base_lr * float(current_step + 1) / float(max(1, warmup_steps))

    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr = base_lr * min_lr_ratio
    return min_lr + (base_lr - min_lr) * cosine_decay

# 13. Training and evaluation helpers

In [ ]:
def top_k_accuracy(outputs, labels, k=5):
    _, top_k_preds = outputs.topk(k, dim=1)
    correct = top_k_preds.eq(labels.view(-1, 1).expand_as(top_k_preds))
    return correct.any(dim=1).float().mean().item()

def run_epoch_with_scheduler(
    model,
    loader,
    criterion,
    optimizer=None,
    phase="Train",
    epoch=1,
    total_epochs=1,
    base_lr=1e-4,
    global_step_start=0,
    total_steps=1,
    warmup_steps=0,
):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = total_top1 = total_top3 = total_top5 = 0
    all_preds, all_labels = [], []
    global_step = global_step_start

    progress_bar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [{phase}]", leave=False)

    with torch.set_grad_enabled(is_train):
        for step, (x, y) in enumerate(progress_bar, start=1):
            x = x.to(device)
            y = y.to(device)

            if is_train:
                lr = cosine_lr(base_lr, global_step, total_steps, warmup_steps)
                set_optimizer_lr(optimizer, lr)
                optimizer.zero_grad()

            outputs = model(x)
            loss = criterion(outputs, y)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRADIENT_CLIP_NORM)
                optimizer.step()
                global_step += 1

            preds = torch.argmax(outputs, dim=1)
            batch_top1 = (preds == y).float().mean().item()
            batch_top3 = top_k_accuracy(outputs, y, 3)
            batch_top5 = top_k_accuracy(outputs, y, 5)

            total_loss += loss.item()
            total_top1 += batch_top1
            total_top3 += batch_top3
            total_top5 += batch_top5

            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(y.detach().cpu().numpy())

            progress_bar.set_postfix({
                "step": f"{step}/{len(loader)}",
                "loss": f"{loss.item():.4f}",
                "top1": f"{batch_top1:.4f}",
                "top5": f"{batch_top5:.4f}",
            })

    avg_loss = total_loss / len(loader)
    avg_top1 = total_top1 / len(loader)
    avg_top3 = total_top3 / len(loader)
    avg_top5 = total_top5 / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return avg_loss, avg_top1, avg_top3, avg_top5, macro_f1, global_step

def top_k_accuracy_numpy(y_true, y_probs, k):
    correct = 0
    for true_label, prob in zip(y_true, y_probs):
        if true_label in np.argsort(prob)[-k:]:
            correct += 1
    return correct / len(y_true)

def collect_predictions(model, loader, temperature=1.0):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Collecting predictions"):
            x = x.to(device)
            outputs = model(x) / temperature
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(y.numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# 14. Stage 1 — freeze feature extractor and train classifier

In [ ]:
stage_1_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=STAGE_1_LR,
    weight_decay=WEIGHT_DECAY,
)

stage_1_total_steps = STAGE_1_EPOCHS * len(train_loader)
stage_1_warmup_steps = max(10, int(0.1 * stage_1_total_steps))

history = []
global_step = 0

print("=" * 80)
print("Stage 1: Frozen feature extractor, train classifier")
print("=" * 80)
print("Trainable parameters:", count_trainable_parameters(model))

for epoch in range(1, STAGE_1_EPOCHS + 1):
    print(f"\nStage 1 Epoch {epoch}/{STAGE_1_EPOCHS}")

    train_loss, train_top1, train_top3, train_top5, train_f1, global_step = run_epoch_with_scheduler(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=stage_1_optimizer,
        phase="Training",
        epoch=epoch,
        total_epochs=STAGE_1_EPOCHS,
        base_lr=STAGE_1_LR,
        global_step_start=global_step,
        total_steps=stage_1_total_steps,
        warmup_steps=stage_1_warmup_steps,
    )

    val_loss, val_top1, val_top3, val_top5, val_f1, _ = run_epoch_with_scheduler(
        model=model,
        loader=val_loader,
        criterion=criterion,
        optimizer=None,
        phase="Validation",
        epoch=epoch,
        total_epochs=STAGE_1_EPOCHS,
    )

    history.append({
        "stage": 1,
        "epoch": epoch,
        "train_loss": train_loss,
        "train_top1": train_top1,
        "train_top3": train_top3,
        "train_top5": train_top5,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_top1": val_top1,
        "val_top3": val_top3,
        "val_top5": val_top5,
        "val_f1": val_f1,
        "lr": stage_1_optimizer.param_groups[0]["lr"],
    })

    print(f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}")

# 15. Stage 2 — unfreeze all layers and fine-tune

In [ ]:
unfreeze_all(model)

stage_2_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=STAGE_2_LR,
    weight_decay=WEIGHT_DECAY,
)

stage_2_total_steps = STAGE_2_EPOCHS * len(train_loader)
stage_2_warmup_steps = max(10, int(0.05 * stage_2_total_steps))

global_step = 0
best_val_f1 = 0.0
best_val_top5 = 0.0
epochs_without_improvement = 0

print("=" * 80)
print("Stage 2: Unfreeze all layers and fine-tune")
print("=" * 80)
print("Trainable parameters:", count_trainable_parameters(model))

start_time = time.time()

for epoch in range(1, STAGE_2_EPOCHS + 1):
    print(f"\nStage 2 Epoch {epoch}/{STAGE_2_EPOCHS}")

    train_loss, train_top1, train_top3, train_top5, train_f1, global_step = run_epoch_with_scheduler(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=stage_2_optimizer,
        phase="Training",
        epoch=epoch,
        total_epochs=STAGE_2_EPOCHS,
        base_lr=STAGE_2_LR,
        global_step_start=global_step,
        total_steps=stage_2_total_steps,
        warmup_steps=stage_2_warmup_steps,
    )

    val_loss, val_top1, val_top3, val_top5, val_f1, _ = run_epoch_with_scheduler(
        model=model,
        loader=val_loader,
        criterion=criterion,
        optimizer=None,
        phase="Validation",
        epoch=epoch,
        total_epochs=STAGE_2_EPOCHS,
    )

    history.append({
        "stage": 2,
        "epoch": epoch,
        "train_loss": train_loss,
        "train_top1": train_top1,
        "train_top3": train_top3,
        "train_top5": train_top5,
        "train_f1": train_f1,
        "val_loss": val_loss,
        "val_top1": val_top1,
        "val_top3": val_top3,
        "val_top5": val_top5,
        "val_f1": val_f1,
        "lr": stage_2_optimizer.param_groups[0]["lr"],
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_top5 = val_top5
        epochs_without_improvement = 0

        checkpoint_payload = {
            "epoch": epoch,
            "stage": 2,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": stage_2_optimizer.state_dict(),
            "best_val_f1": best_val_f1,
            "best_val_top5": best_val_top5,
            "num_classes": NUM_CLASSES,
            "input_size": INPUT_SIZE,
            "sequence_length": SEQUENCE_LENGTH,
            "use_velocity": USE_VELOCITY,
            "architecture": "BiGRUAttentionLightV3",
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
            "training_mode": "two_stage_finetuned_from_wlasl1000",
            "source_checkpoint": str(WLASL1000_MODEL_PATH),
            "loaded_keys": loaded_keys,
            "skipped_keys": skipped_keys,
            "loss": "ClassBalancedFocalLoss",
            "focal_gamma": FOCAL_GAMMA,
            "label_smoothing": LABEL_SMOOTHING,
        }

        torch.save(checkpoint_payload, MODEL_PATH)
        status = "Saved new best Light V3 model"
    else:
        epochs_without_improvement += 1
        status = "No improvement"

    print(f"Train | Loss: {train_loss:.4f} | Top-1: {train_top1:.4f} | Top-3: {train_top3:.4f} | Top-5: {train_top5:.4f} | F1: {train_f1:.4f}")
    print(f"Val   | Loss: {val_loss:.4f} | Top-1: {val_top1:.4f} | Top-3: {val_top3:.4f} | Top-5: {val_top5:.4f} | F1: {val_f1:.4f}")
    print("Status:", status)
    print(f"Best Val F1: {best_val_f1:.4f}")
    print(f"Best Val Top-5: {best_val_top5:.4f}")
    print(f"Patience: {epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nStage 2 training completed.")
print("Training time minutes:", round((time.time() - start_time) / 60, 2))
print("Best model saved:", MODEL_PATH)

# 16. Save history and plot training curves

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print("Saved history:", HISTORY_PATH)
display(history_df.tail())

plt.figure(figsize=(10, 5))
plt.plot(history_df["train_loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("WLASL2000 Light V3 Loss")
plt.xlabel("Epoch index")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history_df["train_top5"], label="Train Top-5")
plt.plot(history_df["val_top5"], label="Validation Top-5")
plt.title("WLASL2000 Light V3 Top-5")
plt.xlabel("Epoch index")
plt.ylabel("Top-5 Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# 17. Evaluate Light V3 on test set

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=device)

model = build_light_v3_model(checkpoint.get("num_classes", NUM_CLASSES)).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_true, y_pred, y_probs = collect_predictions(model, test_loader, temperature=1.0)

test_top1 = accuracy_score(y_true, y_pred)
test_top3 = top_k_accuracy_numpy(y_true, y_probs, 3)
test_top5 = top_k_accuracy_numpy(y_true, y_probs, 5)
test_macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("=" * 80)
print("WLASL2000 Light V3 Test Evaluation")
print("=" * 80)
print(f"Test Top-1 Accuracy: {test_top1:.4f}")
print(f"Test Top-3 Accuracy: {test_top3:.4f}")
print(f"Test Top-5 Accuracy: {test_top5:.4f}")
print(f"Test Macro F1: {test_macro_f1:.4f}")

# 18. Temperature scaling for confidence calibration

In [ ]:
def collect_logits_and_labels(model, loader):
    model.eval()
    logits_list, labels_list = [], []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Collecting validation logits"):
            x = x.to(device)
            logits = model(x)

            logits_list.append(logits.cpu())
            labels_list.append(y)

    return torch.cat(logits_list), torch.cat(labels_list)

val_logits, val_labels = collect_logits_and_labels(model, val_loader)

temperatures = np.linspace(0.5, 3.0, 26)
temp_records = []

for temperature in temperatures:
    scaled_logits = val_logits / float(temperature)
    val_loss = F.cross_entropy(scaled_logits, val_labels).item()
    probs = F.softmax(scaled_logits, dim=1).numpy()
    preds = probs.argmax(axis=1)
    acc = accuracy_score(val_labels.numpy(), preds)

    temp_records.append({
        "temperature": float(temperature),
        "val_nll": val_loss,
        "val_top1_accuracy": acc,
    })

temp_df = pd.DataFrame(temp_records)
best_temperature = float(temp_df.sort_values("val_nll").iloc[0]["temperature"])

temperature_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_temperature_scaling.csv"
temp_df.to_csv(temperature_file, index=False)

print("Saved temperature scaling report:", temperature_file)
print("Best temperature:", best_temperature)

display(temp_df.sort_values("val_nll").head())

# 19. Re-evaluate with calibrated temperature

In [ ]:
y_true_cal, y_pred_cal, y_probs_cal = collect_predictions(
    model,
    test_loader,
    temperature=best_temperature,
)

test_top1_cal = accuracy_score(y_true_cal, y_pred_cal)
test_top3_cal = top_k_accuracy_numpy(y_true_cal, y_probs_cal, 3)
test_top5_cal = top_k_accuracy_numpy(y_true_cal, y_probs_cal, 5)
test_macro_f1_cal = f1_score(y_true_cal, y_pred_cal, average="macro", zero_division=0)

print("=" * 80)
print("WLASL2000 Light V3 Calibrated Test Evaluation")
print("=" * 80)
print(f"Temperature: {best_temperature:.2f}")
print(f"Test Top-1 Accuracy: {test_top1_cal:.4f}")
print(f"Test Top-3 Accuracy: {test_top3_cal:.4f}")
print(f"Test Top-5 Accuracy: {test_top5_cal:.4f}")
print(f"Test Macro F1: {test_macro_f1_cal:.4f}")

# 20. Save Light V3 result summary

In [ ]:
result_df = pd.DataFrame([{
    "dataset": DATASET_NAME,
    "model": MODEL_DISPLAY_NAME,
    "clean_samples": len(df),
    "classes": NUM_CLASSES,
    "test_samples": len(test_df),
    "input_shape": f"(60, {INPUT_SIZE})",
    "features": "keypoints + velocity",
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "checkpoint_epoch": checkpoint["epoch"],
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "calibrated_temperature": best_temperature,
    "calibrated_test_top1_accuracy": test_top1_cal,
    "calibrated_test_top3_accuracy": test_top3_cal,
    "calibrated_test_top5_accuracy": test_top5_cal,
    "calibrated_test_macro_f1": test_macro_f1_cal,
    "model_path": str(MODEL_PATH),
    "history_path": str(HISTORY_PATH),
    "norm_stats_path": str(NORM_STATS_PATH),
    "temperature_report": str(temperature_file),
}])

result_df.to_csv(RESULT_FILE, index=False)

report_result_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_result_summary.csv"
result_df.to_csv(report_result_file, index=False)

print("Saved result summary:")
print(RESULT_FILE)
print(report_result_file)

display(result_df)

# 21. Calibrated confidence threshold analysis

In [ ]:
if LABEL_MAP_FILE.exists():
    with open(LABEL_MAP_FILE, "r", encoding="utf-8") as f:
        label_map = json.load(f)
    id_to_gloss = {int(k): v["gloss"] for k, v in label_map.items()}
else:
    id_to_gloss = {}

test_df_reset = test_df.reset_index(drop=True)
prediction_records = []

for i in range(len(y_true_cal)):
    true_id = int(y_true_cal[i])
    pred_id = int(y_pred_cal[i])
    confidence = float(y_probs_cal[i][pred_id])
    top5_ids = np.argsort(y_probs_cal[i])[-5:][::-1]

    prediction_records.append({
        "video_id": test_df_reset.iloc[i]["video_id"],
        "true_label_id": true_id,
        "true_gloss": id_to_gloss.get(true_id, str(true_id)),
        "predicted_label_id": pred_id,
        "predicted_gloss": id_to_gloss.get(pred_id, str(pred_id)),
        "confidence": confidence,
        "correct_top1": true_id == pred_id,
        "correct_top5": true_id in top5_ids,
        "top5_label_ids": ", ".join([str(int(x)) for x in top5_ids]),
        "top5_glosses": ", ".join([id_to_gloss.get(int(x), str(x)) for x in top5_ids]),
        "top5_probabilities": ", ".join([f"{float(y_probs_cal[i][x]):.4f}" for x in top5_ids]),
    })

predictions_df = pd.DataFrame(prediction_records)

predictions_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_calibrated_test_predictions.csv"
predictions_df.to_csv(predictions_file, index=False)

threshold_records = []

for threshold_value in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    confident = predictions_df[predictions_df["confidence"] >= threshold_value]

    threshold_records.append({
        "confidence_threshold": threshold_value,
        "coverage": len(confident) / len(predictions_df),
        "top1_accuracy_on_confident_samples": confident["correct_top1"].mean() if len(confident) else np.nan,
        "top5_accuracy_on_confident_samples": confident["correct_top5"].mean() if len(confident) else np.nan,
        "num_confident_samples": len(confident),
    })

threshold_df = pd.DataFrame(threshold_records)

threshold_file = REPORT_DIR / f"{MODEL_NAME}_{PREFIX}_calibrated_confidence_threshold_analysis.csv"
threshold_df.to_csv(threshold_file, index=False)

print("Saved calibrated predictions:", predictions_file)
print("Saved calibrated confidence threshold analysis:", threshold_file)

display(threshold_df)

# 22. Compare Light V2 vs Light V3

In [ ]:
v2_file = MODEL_DIR / f"light_v2_finetuned_from_wlasl1000_{PREFIX}_result_summary.csv"

comparison_rows = []

if v2_file.exists():
    v2 = pd.read_csv(v2_file).iloc[0].to_dict()

    comparison_rows.append({
        "version": "Light V2",
        "model": v2.get("model", "WLASL2000 Light V2 Fine-tuned From WLASL1000"),
        "test_top1_accuracy": float(v2.get("test_top1_accuracy", np.nan)),
        "test_top3_accuracy": float(v2.get("test_top3_accuracy", np.nan)),
        "test_top5_accuracy": float(v2.get("test_top5_accuracy", np.nan)),
        "test_macro_f1": float(v2.get("test_macro_f1", np.nan)),
        "best_val_f1": float(v2.get("best_val_f1", np.nan)),
        "best_val_top5": float(v2.get("best_val_top5", np.nan)),
        "source": str(v2_file),
    })
else:
    print("Previous Light V2 result not found:", v2_file)

comparison_rows.append({
    "version": "Light V3",
    "model": MODEL_DISPLAY_NAME,
    "test_top1_accuracy": test_top1,
    "test_top3_accuracy": test_top3,
    "test_top5_accuracy": test_top5,
    "test_macro_f1": test_macro_f1,
    "best_val_f1": checkpoint["best_val_f1"],
    "best_val_top5": checkpoint["best_val_top5"],
    "source": str(RESULT_FILE),
})

comparison_df = pd.DataFrame(comparison_rows)

comparison_file = REPORT_DIR / f"{PREFIX}_light_v2_vs_light_v3_comparison.csv"
comparison_df.to_csv(comparison_file, index=False)

print("Saved comparison:", comparison_file)

display(comparison_df)

# 23. Final recommendation

In [ ]:
if len(comparison_df) >= 2:
    v2_top5 = comparison_df[comparison_df["version"] == "Light V2"]["test_top5_accuracy"].iloc[0]
    v3_top5 = comparison_df[comparison_df["version"] == "Light V3"]["test_top5_accuracy"].iloc[0]

    v2_top1 = comparison_df[comparison_df["version"] == "Light V2"]["test_top1_accuracy"].iloc[0]
    v3_top1 = comparison_df[comparison_df["version"] == "Light V3"]["test_top1_accuracy"].iloc[0]

    print("Model decision")
    print("--------------")

    if (v3_top5 > v2_top5) and (v3_top1 >= v2_top1 * 0.95):
        print("Recommended: Use Light V3 as the new app candidate.")
        print("Reason: Light V3 improved Top-5 without damaging Top-1 too much.")
    elif v3_top1 > v2_top1:
        print("Recommended: Use Light V3 as the new app candidate.")
        print("Reason: Light V3 improved Top-1.")
    else:
        print("Recommended: Keep Light V2 as the app candidate for now.")
        print("Reason: Light V3 did not clearly improve the deployment metrics.")

print()
print("For app deployment, do not auto-speak yet.")
print("Recommended behaviour:")
print("confidence >= 0.50 → show Top-1 as main suggestion + show Top-5")
print("0.30 <= confidence < 0.50 → show Top-5 suggestions")
print("confidence < 0.30 → ask user to sign again")